# CTDDG | MXNet checkpoint evaluation
### Chemical validity, Ames mutagenicity & drug-likeness

A **step-by-step, read-only** evaluation notebook for the original MXNet pretrained checkpoint in
`/home/muhamed/repo/CTDDG/satyajit_repo/model`.

**What you will see:** checkpoint/vocabulary checks → generation source → raw attempt counts → RDKit validity and failure reasons → uniqueness → Random Forest Ames predictions → physicochemical properties → per-run/protein tables → saved outputs. Every few cells print or display results.

**Important distinction:** an MXNet `.params` checkpoint, `configs.json` and `atom_types.txt` **do not uniquely specify how to sample graph actions**. The original repository's matching model class and graph decoder must be used to generate fresh molecules. The `GENERATE_WITH_REPO` mode below invokes **your verified repository sampling command**, while `EXISTING_MXNET_CSV` scores its already-generated results. We do **not** invent a model architecture or silently use a PyTorch checkpoint.

**Run on Bhavani** where those paths exist, preferably inside the appropriate scheduled compute allocation for GPU inference. No training, file overwrites or package upgrades occur in this notebook.

## 1. Protocol and metric definitions

The **CTDDG** paper uses **1,042 distinct training proteins** and **104 distinct held-out test proteins** for its *protein-conditioned fine-tuned* model. It reports 100 samples per test protein over 10 runs. The pretrained MXNet checkpoint supplied here may be **unconditional**; unless its metadata and code show a protein-conditioning branch, its evaluation is explicitly **pretraining-only** and cannot be compared directly with the 104-protein table.

Following `Evaluation_metrics.ipynb` (Ma'am's notebook), for a requested budget of `N` samples:

- **Validity (%)** = `100 × number of RDKit-valid generated molecules / N`.
- **Uniqueness vs requested (%)** = `100 × distinct canonical valid molecules / N` (the original notebook's denominator).
- Also report **uniqueness among valid (%)** = `100 × unique valid / valid`, to avoid confusion.
- **Predicted Ames-positive (%)** = `100 × RF-predicted class 1 / RF-scored valid molecules`. This is *not* the pkCSM Ames percentage in CTDDG Table 3.
- LogP, ExactMolWt, H-donors/acceptors, rotatable bonds, TPSA, QED, and SAS follow the reference notebook's RDKit methods. The reference also computes unusual *uniqueness-weighted* pass rates; we show those as explicitly named auxiliary columns, **not** as ordinary pass rates.

If a generator exports only valid molecules, the missing `N - exported_rows` are included in the **requested-size** denominator but their failure reasons are **unknown**. To diagnose failure reasons, supply an all-attempts CSV with invalid/empty rows.

## 2. Configuration — edit this cell first

The **MXNet** paths below are your supplied HPC paths. The Random Forest path and repository root come from your *PyTorch Generation and Toxicity Evaluation* notebook, but **the PyTorch weights are not used**.

- `SOURCE_MODE = 'EXISTING_MXNET_CSV'`: use an MXNet generator CSV or its output folder, **not** your earlier PyTorch CSV.
- `SOURCE_MODE = 'GENERATE_WITH_REPO'`: run the model's **actual MXNet sampling CLI** after filling `SAMPLER_COMMAND` with its verified arguments. It must save generated SMILES to `GENERATED_SOURCE`.

The checkpoint inspector runs in either mode. The default file source is left unset on purpose to prevent quietly evaluating the wrong model's output.

In [1]:
from pathlib import Path
from datetime import datetime

ROOT = Path('/home/muhamed/repo/CTDDG')
MODEL_DIR = ROOT / 'satyajit_repo/model'
MXNET_WEIGHTS = MODEL_DIR / 'ckpt.params'
ATOM_TYPES_PATH = MODEL_DIR / 'atom_types.txt'
CONFIG_PATH = MODEL_DIR / 'configs.json'
RF_MODEL_PATH = ROOT / 'REWORKED_CODE/Random_Forest/RF_NItesh.joblib'

SOURCE_MODE = 'EXISTING_MXNET_CSV'  # or 'GENERATE_WITH_REPO'
GENERATED_SOURCE = None            # Path('/.../actual_mxnet_samples.csv') or a folder
SAMPLER_COMMAND = []              # verified argv only, no guessed flags; see Section 6

# Paper-style evaluation: 100 samples × 104 test proteins × 10 independent runs.
# For a first unconditional PRETRAINING test, use just 100 attempts × 1 run.
SAMPLES_PER_PROTEIN = 100
REPETITIONS = 1                 # set 10 for full repeated evaluation
EXPECTED_TEST_PROTEINS = 104
EXPECTED_TRAIN_PROTEINS = 1042
MODEL_STAGE = 'pretrained_unconditional'  # do not change until verified in model code
RANDOM_SEED = 42
RF_POSITIVE_CLASS = 1
MORGAN_RADIUS = 2
MORGAN_BITS = 2048
DRAW_N = 9
REQUIRE_RF = True

# Optional: provide *real* BindingDB split CSVs (to verify distinct sequence counts).
BINDINGDB_TRAIN_CSV = None
BINDINGDB_TEST_CSV = None
# Optional: exported pkCSM scores on these exact molecules, if independently obtained.
PKCSM_AMES_CSV = None

EVAL_TAG = 'mxnet_eval_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT_DIR = ROOT / 'evaluation_results' / EVAL_TAG
print('Checkpoint:', MXNET_WEIGHTS)
print('Generation mode:', SOURCE_MODE)
print('Model stage:', MODEL_STAGE)
print('Output directory (new, unique):', OUT_DIR)

Checkpoint: /home/muhamed/repo/CTDDG/satyajit_repo/model/ckpt.params
Generation mode: EXISTING_MXNET_CSV
Model stage: pretrained_unconditional
Output directory (new, unique): /home/muhamed/repo/CTDDG/evaluation_results/mxnet_eval_20261106_213057_205509


## 3. Environment and dependencies

Inspect versions before proceeding. MXNet availability is diagnosed separately so chemical evaluation can still be tested on exported CSVs in a different Python environment.

In [2]:
import sys, os, json, hashlib, re, subprocess, statistics, time
from collections import Counter
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from rdkit import Chem, RDLogger, DataStructs, rdBase
from rdkit.Chem import (AllChem, Descriptors, Crippen, QED, Lipinski, Draw)
import sklearn
import matplotlib.pyplot as plt

try:
    import mxnet as mx
    MXNET_READY = True
    MXNET_ERROR = None
except Exception as exc:
    mx, MXNET_READY, MXNET_ERROR = None, False, f'{type(exc).__name__}: {exc}'

print('Python:', sys.executable)
print('RDKit:', rdBase.rdkitVersion, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)
print('MXNet:', mx.__version__ if MXNET_READY else 'UNAVAILABLE: ' + MXNET_ERROR)
print('MXNet is needed to INSPECT the .params file, and the exact repo code is needed to GENERATE.')

Python: /home/muhamed/.conda/envs/ctddg_torch/bin/python
RDKit: 2026.03.6 | pandas: 3.0.6 | sklearn: 1.9.1
MXNet: UNAVAILABLE: ModuleNotFoundError: No module named 'mxnet'
MXNet is needed to INSPECT the .params file, and the exact repo code is needed to GENERATE.


## 4. Confirm the exact files and their provenance

Checking model metadata is particularly important because an unconditional pretraining checkpoint should not be called a protein-conditioned 104-protein model. These checks never alter the checkpoint.

In [3]:
for name, path in [('MXNet checkpoint', MXNET_WEIGHTS), ('Atom vocabulary', ATOM_TYPES_PATH),
                   ('Model config', CONFIG_PATH), ('Ames Random Forest', RF_MODEL_PATH)]:
    isfile = path.is_file()
    print(f'{name:22} {"FOUND" if isfile else "MISSING":7}  '
          f'{path.stat().st_size/1024**2:.2f} MiB' if isfile else f'{name:22} MISSING  {path}')

missing = [str(p) for p in (MXNET_WEIGHTS, ATOM_TYPES_PATH, CONFIG_PATH) if not p.is_file()]
if missing:
    raise FileNotFoundError('Run this notebook ON BHAVANI. Missing model files: ' + str(missing))
if REQUIRE_RF and not RF_MODEL_PATH.is_file():
    print('WARNING: RF missing; validity will still be evaluable, but toxicity cannot be reported.')

MXNet checkpoint       FOUND    30.88 MiB
Atom vocabulary        FOUND    0.00 MiB
Model config           FOUND    0.00 MiB
Ames Random Forest     FOUND    21.92 MiB


In [4]:
def sha256_file(path, chunk_bytes=1024*1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(chunk_bytes), b''):
            h.update(block)
    return h.hexdigest()

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
print('Model config top-level type:', type(config).__name__)
if isinstance(config, dict):
    display(pd.DataFrame([{'key':k,'value_preview':repr(v)[:180]} for k,v in config.items()]).head(30))
else:
    print('Config preview:', repr(config)[:500])
print('Weights SHA256:', sha256_file(MXNET_WEIGHTS))
print('Vocabulary SHA256:', sha256_file(ATOM_TYPES_PATH))

Model config top-level type: dict


,key,value_preview
0,N_C,1024
1,F_e,16
2,F_h,"[32, 64, 128, 128, 256, 256]"
3,F_skip,256
4,F_c,[512]
5,Fh_policy,128
6,activation,'relu'
7,rename,False
8,N_rnn,3


Weights SHA256: d7055224a850cb8ecb29de2f58f1daae4e8ef7c7a6184d78a1a6535bb430190b
Vocabulary SHA256: 9e45837809d98240490ad0ae25cbe0a57117bc83e68b7c1d01902d4cc6e26cf0


## 5. Inspect the ordered vocabulary and MXNet tensors

The original checkpoints' ordered atom types matter for graph decoding. We validate their parsing, then inspect MXNet parameter names and shapes. `mx.nd.load(...)` **only inspects tensors**; it does **not** instantiate the graph generative model or sample molecules.

In [5]:
atom_types = []
for line_number, line in enumerate(ATOM_TYPES_PATH.read_text().splitlines(), 1):
    if not line.strip():
        continue
    parts = [p.strip() for p in line.split(',')]
    if len(parts) != 3:
        raise ValueError(f'Unexpected atom_types.txt format on line {line_number}: {line!r}')
    atom_types.append((parts[0], int(parts[1]), int(parts[2])))
assert atom_types and len(atom_types) == len(set(atom_types)), 'Empty or duplicate atom types.'
print('Number of ordered atom types:', len(atom_types))
display(pd.DataFrame(atom_types[:12], columns=['symbol','formal_charge','explicit_hydrogens']))
print('NOTE: this must match the vocabulary order used when the checkpoint was trained.')

Number of ordered atom types: 67


,symbol,formal_charge,explicit_hydrogens
0,P,0,0
1,F,0,0
2,O,0,1
3,Sn,0,0
4,K,0,0
5,S,0,2
6,Fe,-2,0
7,Cl,3,0
8,Zn,2,0
9,S,-1,0


NOTE: this must match the vocabulary order used when the checkpoint was trained.


In [6]:
if not MXNET_READY:
    print('SKIPPED MXNet tensor inspection:', MXNET_ERROR)
else:
    mx_weights = mx.nd.load(str(MXNET_WEIGHTS))
    if isinstance(mx_weights, dict):
        shape_rows = [{'tensor_name':str(k), 'shape':tuple(v.shape), 'dtype':str(v.dtype)}
                      for k,v in mx_weights.items()]
    elif isinstance(mx_weights, (list, tuple)):
        shape_rows = [{'tensor_name':f'array_{i}', 'shape':tuple(v.shape), 'dtype':str(v.dtype)}
                      for i,v in enumerate(mx_weights)]
    else:
        raise TypeError('Unrecognized MXNet checkpoint format: ' + type(mx_weights).__name__)
    print('Checkpoint tensor count:', len(shape_rows))
    display(pd.DataFrame(shape_rows).head(20))
    print('NOTE: not yet a forward pass. A matching model/decoder is required for generation.')
    del mx_weights

SKIPPED MXNet tensor inspection: ModuleNotFoundError: No module named 'mxnet'


## 6. Locate the *real* MXNet generator / load saved generations

Your uploaded files do not include the repository's model-construction and generation Python scripts. **Do not guess an MXNet architecture from the three model files.** This cell lists likely source files to help you identify the real inference entry point. It does not run unknown scripts.

**Option A — already generated MXNet CSV:** Set `GENERATED_SOURCE` to its CSV, or to a folder of `Protein1_generated_samples.csv`, etc. This is the quickest route to validity and toxicity.

**Option B — generate from weights:** Find the repository's actual sampling command, populate `SAMPLER_COMMAND` (a list of command-line tokens) in configuration, and set `SOURCE_MODE = 'GENERATE_WITH_REPO'`. It should use `ckpt.params`, `configs.json`, `atom_types.txt`, and write CSVs with a `smiles` column. For a paper-style 104-protein evaluation, its generator must also consume the **actual held-out test protein sequences**, not training proteins.

In [7]:
repo_base = ROOT / 'satyajit_repo'
keywords = ('generat','sampl','infer','predict')
repo_candidates = sorted([p for p in repo_base.rglob('*.py')
                          if any(k in p.name.lower() for k in keywords)]) if repo_base.exists() else []
print('Repository sampler candidates (review their code and --help first):')
for p in repo_candidates[:30]:
    print(' -', p.relative_to(ROOT))
print('Candidates found:', len(repo_candidates))
print('You can set GENERATED_SOURCE to an existing MXNet samples directory without rerunning inference.')

Repository sampler candidates (review their code and --help first):
Candidates found: 0
You can set GENERATED_SOURCE to an existing MXNet samples directory without rerunning inference.


In [8]:
# This cell executes no command unless SOURCE_MODE explicitly requests it.
if SOURCE_MODE not in {'EXISTING_MXNET_CSV', 'GENERATE_WITH_REPO'}:
    raise ValueError('SOURCE_MODE must be EXISTING_MXNET_CSV or GENERATE_WITH_REPO.')
if SOURCE_MODE == 'GENERATE_WITH_REPO':
    if not MXNET_READY:
        raise RuntimeError('This notebook kernel cannot import MXNet: ' + str(MXNET_ERROR))
    if not SAMPLER_COMMAND:
        raise RuntimeError('Set SAMPLER_COMMAND to the VERIFIED original MXNet generation CLI. '
                           'The supplied notebooks do not contain the MXNet sampler implementation. '
                           'Do not use a guessed CLI or a PyTorch generator.')
    assert all(isinstance(t, str) for t in SAMPLER_COMMAND), 'SAMPLER_COMMAND must be list[str].'
    print('Running verified MXNet sampler:', ' '.join(SAMPLER_COMMAND))
    completed = subprocess.run(SAMPLER_COMMAND, cwd=str(ROOT), check=True, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout[-12000:])
else:
    print('Re-using existing MXNet generator outputs (no inference executed).')
print('CSV source configuration:', GENERATED_SOURCE)

Re-using existing MXNet generator outputs (no inference executed).
CSV source configuration: None


## 7. Read generation output — **no silent PyTorch fallback**

The loader accepts a CSV or a folder of CSVs. It can read Ma'am's hierarchy
`.../100/run1/Protein1_generated_samples.csv`, or a flat all-attempt CSV with `smiles`, `run`, `protein_id`, and `attempt` columns.

The `protein_id` and `run` identifiers are inferred from **filenames** only when absent from the CSV. Do not use the PyTorch example's `generated_run1.csv` here; it would evaluate a different checkpoint.

In [9]:
def find_generated_csvs(source):
    if source is None:
        raise ValueError('GENERATED_SOURCE is unset. Point it at CSVs produced by the ORIGINAL MXNet '
                         'sampler using these pretrained weights. The three model files alone do not '
                         'contain a sampler. See Section 6.')
    p = Path(source).expanduser()
    if p.is_file():
        if p.suffix.lower() != '.csv':
            raise ValueError('Source file must be a CSV.')
        return [p]
    if p.is_dir():
        matches = sorted(p.rglob('*.csv'))
        matches = [x for x in matches if 'summary' not in x.name.lower()
                   and 'fingerprint' not in x.name.lower() and 'metric' not in x.name.lower()]
        if matches: return matches
    raise FileNotFoundError(f'No generated-molecule CSV files at {p}.')

files = find_generated_csvs(GENERATED_SOURCE)
print('Number of generated CSV files:', len(files))
for p in files[:8]: print(' -', p)

ValueError: GENERATED_SOURCE is unset. Point it at CSVs produced by the ORIGINAL MXNet sampler using these pretrained weights. The three model files alone do not contain a sampler. See Section 6.

In [ ]:
def metadata_from_name(p):
    full = str(p)
    m_run = re.search(r'(?:^|/)run[_-]?(\d+)(?:/|$)', full, re.I)
    m_pro = re.search(r'protein[_-]?(\d+)', p.name, re.I)
    m_n = re.search(r'(?:^|/)(10|100|1000)(?:/|$)', full)
    return (int(m_run.group(1)) if m_run else 1,
            str(int(m_pro.group(1))) if m_pro else 'unconditional',
            int(m_n.group(1)) if m_n else SAMPLES_PER_PROTEIN)

parts = []
for path in files:
    temp = pd.read_csv(path)
    if temp.empty and not len(temp.columns):
        continue
    col_map = {str(col).lower().strip():col for col in temp.columns}
    smi_col = next((col_map[k] for k in ('smiles','smile','generated_smiles','canonical_smiles') if k in col_map), None)
    if smi_col is None:
        print('SKIP (no smiles column):', path.name, list(temp.columns)[:10]); continue
    default_run, default_prot, default_n = metadata_from_name(path)
    temp = temp.copy().rename(columns={smi_col:'smiles'})
    if 'run' not in temp: temp['run'] = default_run
    if 'protein_id' not in temp: temp['protein_id'] = default_prot
    if 'requested_n' not in temp: temp['requested_n'] = default_n
    if 'attempt' not in temp: temp['attempt'] = np.arange(1,len(temp)+1)
    temp['source_file'] = str(path)
    parts.append(temp)
assert parts, 'No usable generated samples CSV found.'
raw = pd.concat(parts, ignore_index=True, sort=False)
raw['run'] = raw['run'].astype(str)
raw['protein_id'] = raw['protein_id'].astype(str)
raw['requested_n'] = pd.to_numeric(raw['requested_n'], errors='raise').astype(int)
assert (raw['requested_n'] > 0).all()
print('Imported attempt rows:', len(raw))
print('Runs:', raw['run'].nunique(), '| distinct protein IDs:', raw['protein_id'].nunique())
display(raw[['run','protein_id','attempt','smiles','source_file']].head(10))

## 8. Verify attempt counts **before** chemical checks

Ma'am's validity calculation uses the requested sample count (`beam_size`) as denominator, so we check it *per protein and per run* before evaluating a single molecule. Missing outputs are not the same as RDKit sanitization failures.

In [ ]:
group_cols = ['run','protein_id','source_file']
counts = (raw.groupby(group_cols, dropna=False)
          .agg(imported_rows=('smiles','size'), requested_n=('requested_n','first'))
          .reset_index())
assert not (counts.imported_rows > counts.requested_n).any(), (
    'A file has more rows than its declared sample budget. Inspect counts before continuing.')
counts['not_exported'] = counts.requested_n - counts.imported_rows
print('Requests represented in CSV:', int(counts.requested_n.sum()))
print('Exported attempts:', int(counts.imported_rows.sum()))
print('Missing from exported CSV:', int(counts.not_exported.sum()))
display(counts[['run','protein_id','imported_rows','requested_n','not_exported']].head(15))

## 9. RDKit chemical validity — same core idea as the reference notebook

`Chem.MolFromSmiles(..., sanitize=True)` checks whether each generated SMILES can be parsed and chemically sanitized. Store both valid and invalid rows. In a file with prefiltered valid molecules, the missing attempts still reduce **validity / requested**.

In [ ]:
RDLogger.DisableLog('rdApp.error')   # error details are captured in the table below

def validate_smiles(value):
    if value is None or pd.isna(value) or not str(value).strip():
        return 0, None, 'empty_or_missing_smiles', ''
    s = str(value).strip()
    try:
        mol = Chem.MolFromSmiles(s, sanitize=True)
        if mol is None:
            return 0, None, 'rdkit_parse_or_sanitization_failed', ''
        return 1, Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True), '', ''
    except Exception as exc:
        return 0, None, 'rdkit_exception', f'{type(exc).__name__}: {exc}'

# Sanity checks on deliberately valid / invalid examples.
assert validate_smiles('CCO')[0] == 1
assert validate_smiles('C1CC')[0] == 0
assert validate_smiles('')[0] == 0
print('PASS: RDKit validity checks classify ethanol, unclosed ring, and blank correctly.')

In [ ]:
t0 = time.perf_counter()
validity_rows = [validate_smiles(v) for v in raw['smiles']]
evaluated = raw.copy()
evaluated[['valid','canonical_smiles','invalid_reason','rdkit_error']] = pd.DataFrame(
    validity_rows, index=evaluated.index)
evaluated['valid'] = evaluated['valid'].astype(int)
print('RDKit check complete in', round(time.perf_counter()-t0,2), 'seconds')
print('RDKit valid exported rows:', int(evaluated.valid.sum()), '/', len(evaluated))
print('Invalid exported row reasons:')
display(evaluated.loc[evaluated.valid.eq(0),'invalid_reason'].value_counts().to_frame('count'))
display(evaluated[['run','protein_id','smiles','canonical_smiles','valid','invalid_reason']].head(12))

In [ ]:
valid_rows = evaluated.loc[evaluated.valid == 1].copy()
print('Valid, canonical molecules:', len(valid_rows))
print('Distinct valid canonical molecules overall:', valid_rows['canonical_smiles'].nunique())
if len(valid_rows):
    chosen = valid_rows[['canonical_smiles']].drop_duplicates().head(DRAW_N)
    mols_draw = [Chem.MolFromSmiles(x) for x in chosen['canonical_smiles']]
    display(Draw.MolsToGridImage(mols_draw, molsPerRow=3, subImgSize=(230,175),
                                 legends=[str(i+1) for i in range(len(mols_draw))]))
else:
    print('No RDKit-valid outputs. Inspect generator and vocabulary; do not run toxicity scoring.')

## 10. Validity and uniqueness, **per run and protein**

This gives precisely the reference-style `% valid / requested`. It also reports the complementary conditional metric `% unique / valid`.

In [ ]:
def validity_group(group):
    n_req = int(group['requested_n'].iloc[0])
    n_valid = int(group.valid.sum())
    n_unique = int(group.loc[group.valid == 1, 'canonical_smiles'].nunique())
    return pd.Series(dict(requested=n_req, exported=len(group), valid=n_valid,
                          unique_valid=n_unique,
                          validity_pct=100*n_valid/n_req,
                          unique_requested_pct=100*n_unique/n_req,
                          unique_valid_pct=100*n_unique/n_valid if n_valid else np.nan))

validity_table = (evaluated.groupby(group_cols, sort=True).apply(validity_group)
                  .reset_index())
assert (validity_table.validity_pct <= 100.00001).all()
print('Ref-style validity and uniqueness:')
display(validity_table[['run','protein_id','requested','exported','valid',
                         'validity_pct','unique_valid','unique_requested_pct','unique_valid_pct']].head(20)
        .round(2))
print('Mean validity across represented protein-run groups:',
      f'{validity_table.validity_pct.mean():.2f}%')

## 11. Load the Ames mutagenicity predictor

The CTDDG manuscript uses **pkCSM** to report Ames toxicity (Table 3). Your other notebook instead points to `RF_NItesh.joblib`. We reuse *that exact file* and its **Morgan radius 2 / 2,048-bit** fingerprint method; this is an **RF surrogate Ames prediction**, not a pkCSM prediction or a wet-lab Ames test. The training label provenance must be independently verified before publication.

In [ ]:
rf = None
if RF_MODEL_PATH.is_file():
    rf = joblib.load(RF_MODEL_PATH)   # joblib uses pickle; load only trusted local artifacts
    print('Estimator:', type(rf).__name__)
    print('Classes:', list(rf.classes_))
    print('Model input dimension:', getattr(rf,'n_features_in_', '<unknown>'))
    assert int(rf.n_features_in_) == MORGAN_BITS, 'RF fingerprint width mismatch; STOP.'
    assert set(np.asarray(rf.classes_).astype(int)) == {0,1}, 'Unexpected Ames class encoding; STOP.'
    assert RF_POSITIVE_CLASS in list(rf.classes_), 'Missing positive class; STOP.'
    print('PASS: RF feature dimensions and class labels verified.')
elif REQUIRE_RF:
    raise FileNotFoundError('Ames RF is required for toxicity scoring: ' + str(RF_MODEL_PATH))
else:
    print('RF unavailable; toxicity fields will remain not evaluated, never set to zero.')

## 12. Generate Morgan fingerprints (shown separately)

The fingerprint is calculated **only for chemically valid molecules**. The exact bit-vector size must match the classifier's `n_features_in_`. Molecular graphs that fail RDKit are not given a fake non-toxic label.

In [ ]:
def morgan_bits(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: raise ValueError('Cannot fingerprint invalid SMILES.')
    vec = AllChem.GetMorganFingerprintAsBitVect(mol, MORGAN_RADIUS, nBits=MORGAN_BITS)
    arr = np.zeros(MORGAN_BITS, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(vec, arr)
    return arr

if rf is not None and len(valid_rows):
    fps = [morgan_bits(s) for s in valid_rows.canonical_smiles]
    X_ames = np.stack(fps)
    assert X_ames.shape == (len(valid_rows), MORGAN_BITS)
    print('RF feature matrix:', X_ames.shape, 'dtype=', X_ames.dtype)
    print('Active fingerprint bits min/median/max:',
          int(X_ames.sum(1).min()), float(np.median(X_ames.sum(1))), int(X_ames.sum(1).max()))
else:
    X_ames = np.empty((0,MORGAN_BITS), dtype=np.uint8)
    print('No fingerprints built:', 'RF unavailable' if rf is None else 'no valid molecules')

## 13. Predict Ames labels **and** probabilities

Use `rf.predict` for class labels and the column corresponding to class `1` from `rf.predict_proba` for the estimated positive-class probability. Probabilities must lie in `[0,1]`. **No probabilities are summed across atoms or multiplied by molecule size.**

In [ ]:
evaluated['ames_prediction'] = pd.Series(pd.array([pd.NA]*len(evaluated),dtype='Int64'), index=evaluated.index)
evaluated['ames_positive_probability'] = np.nan
if rf is not None and len(X_ames):
    y_hat = np.asarray(rf.predict(X_ames)).astype(int)
    prob_matrix = np.asarray(rf.predict_proba(X_ames),dtype=float)
    pos_index = list(rf.classes_).index(RF_POSITIVE_CLASS)
    proba = prob_matrix[:,pos_index]
    assert prob_matrix.shape == (len(X_ames),2)
    assert np.isfinite(prob_matrix).all()
    assert (prob_matrix >= 0).all() and (prob_matrix <= 1).all()
    assert np.allclose(prob_matrix.sum(axis=1),1.0,atol=1e-5)
    evaluated.loc[valid_rows.index, 'ames_prediction'] = y_hat
    evaluated.loc[valid_rows.index, 'ames_positive_probability'] = proba
    print('RF class counts:', dict(Counter(y_hat.tolist())))
    print('Ames-positive probability range:', round(float(proba.min()),4), 'to',round(float(proba.max()),4))
    display(evaluated.loc[valid_rows.index,
        ['canonical_smiles','ames_prediction','ames_positive_probability']].head(12))
else:
    print('No Ames predictions (missing RF or no valid samples).')
assert evaluated.loc[evaluated.valid==0,'ames_prediction'].isna().all()

## 14. Toxicity percentages with an explicit denominator

Report three values separately: Ames-positive amongst RF-scored valid molecules (main result), Ames-negative amongst scored molecules, and Ames-positive amongst all requested attempts (a **lower bound** when invalid/unscored outputs exist). Invalid outputs have **unknown** toxicity and are never assumed non-toxic.

In [ ]:
scored = evaluated.loc[evaluated.ames_prediction.notna()].copy()
n_scored = len(scored)
n_pos = int(scored.ames_prediction.eq(1).sum())
n_neg = int(scored.ames_prediction.eq(0).sum())
n_requested = int(validity_table.requested.sum())
print('Requested molecule generation attempts:', n_requested)
print('Chemically valid molecules:', len(valid_rows))
print('RF-scored valid molecules:', n_scored)
print('Predicted Ames-positive:', n_pos, '| predicted Ames-negative:', n_neg)
print('Predicted Ames-positive among scored:',
      f'{100*n_pos/n_scored:.2f}%' if n_scored else 'NOT EVALUATED')
print('Predicted Ames-negative among scored:',
      f'{100*n_neg/n_scored:.2f}%' if n_scored else 'NOT EVALUATED')
print('Ames-positive among all requested (lower bound):',
      f'{100*n_pos/n_requested:.2f}%' if n_scored else 'NOT EVALUATED')

In [ ]:
if len(scored):
    print('Predicted Ames-positive examples:')
    display(scored.loc[scored.ames_prediction.eq(1),
           ['protein_id','canonical_smiles','ames_positive_probability']].head(12))
    print('Estimated probability distribution:')
    display(scored.ames_positive_probability.describe().to_frame('P(Ames+)'))
else:
    print('No scored molecules to display.')

## 14A. Optional **pkCSM Ames** cross-check (paper's toxicity source)

The CTDDG Table 3 toxicity numbers are from the **pkCSM Ames mutagenicity** web server, not the Random Forest. If you have independently obtained pkCSM CSV predictions for these exact generated molecules, set `PKCSM_AMES_CSV` to a file with columns `smiles` and `ames_mutagenicity` (values `Yes/No` or `1/0`). This cell keeps pkCSM and RF separate and shows their agreement. It does not call a web server, invent pkCSM predictions, or silently relabel unknown values.

In [ ]:
if PKCSM_AMES_CSV is None:
    print('pkCSM comparison: not supplied. RF Ames numbers are not the paper’s pkCSM numbers.')
else:
    pk = pd.read_csv(PKCSM_AMES_CSV)
    assert {'smiles','ames_mutagenicity'}.issubset(pk.columns), 'Provide smiles and ames_mutagenicity columns.'
    value_map = {'yes':1, 'no':0, '1':1, '0':0, 'positive':1, 'negative':0,
                 'mutagenic':1,'non-mutagenic':0, 'true':1, 'false':0}
    pk['pkcsm_ames'] = pk.ames_mutagenicity.astype(str).str.strip().str.lower().map(value_map)
    assert pk.pkcsm_ames.notna().all(), 'Unknown pkCSM label encountered; inspect before interpreting.'
    pk['canonical_smiles'] = [validate_smiles(s)[1] for s in pk.smiles]
    assert pk.canonical_smiles.notna().all(), 'pkCSM input contains unparseable SMILES.'
    disagreements = pk.groupby('canonical_smiles').pkcsm_ames.nunique()
    assert disagreements.max() <= 1, 'Conflicting pkCSM labels for same molecule.'
    key = pk.drop_duplicates('canonical_smiles').set_index('canonical_smiles').pkcsm_ames
    evaluated['pkcsm_ames'] = evaluated.canonical_smiles.map(key).astype('Int64')
    sub = evaluated.loc[evaluated.pkcsm_ames.notna()]
    print('pkCSM-scored generated molecules:',len(sub),'/',int(evaluated.valid.sum()))
    print('pkCSM Ames-positive %:', round(100*sub.pkcsm_ames.eq(1).mean(),2) if len(sub) else 'N/A')
    comparable = sub.loc[sub.ames_prediction.notna()]
    print('RF / pkCSM agreement on matched molecules:',
          round(100*(comparable.pkcsm_ames == comparable.ames_prediction).mean(),2) if len(comparable) else 'N/A')
    if len(comparable):
        display(pd.crosstab(comparable.ames_prediction,comparable.pkcsm_ames,
                             rownames=['RF Ames'], colnames=['pkCSM Ames']))

## 15. Physicochemical and drug-likeness properties

Use the exact **family of RDKit functions** from `Evaluation_metrics.ipynb`: `Descriptors.ExactMolWt`, `Crippen.MolLogP`, `QED.qed`, `Lipinski.NumHDonors`, `NumHAcceptors`, `NumRotatableBonds`, and `Descriptors.TPSA`. SAS uses RDKit's bundled Ertl/Landrum SA-score (`sascorer`), equivalent in method to the long custom function in Ma'am's notebook. If SAS support is missing, the result stays **unavailable**, not zero.

In [ ]:
try:
    from rdkit.Contrib.SA_Score import sascorer
    SAS_READY = True
    print('SAS score implementation: RDKit Contrib SA_Score.sascorer')
except ImportError as exc:
    SAS_READY = False
    print('SAS UNAVAILABLE:', exc)

def properties_of_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: raise ValueError('Expected RDKit-valid canonical SMILES.')
    return dict(MW=Descriptors.ExactMolWt(mol), logP=Crippen.MolLogP(mol),
                QED=QED.qed(mol), HD=Lipinski.NumHDonors(mol),
                HA=Lipinski.NumHAcceptors(mol),
                RB=Lipinski.NumRotatableBonds(mol),
                TPSA=Descriptors.TPSA(mol),
                SAS=sascorer.calculateScore(mol) if SAS_READY else np.nan)

print('Example (ethanol):', {k:round(float(v),3) for k,v in properties_of_smiles('CCO').items()})

In [ ]:
prop_columns = ['MW','logP','QED','HD','HA','RB','TPSA','SAS']
for c in prop_columns: evaluated[c] = np.nan
if len(valid_rows):
    # De-duplicate computations; preserve each original attempt for fair denominators.
    unique_smiles = valid_rows.canonical_smiles.unique()
    prop_lookup = {s:properties_of_smiles(s) for s in unique_smiles}
    properties_df = pd.DataFrame([prop_lookup[s] for s in valid_rows.canonical_smiles],
                                  index=valid_rows.index)
    for col in prop_columns: evaluated.loc[valid_rows.index,col] = properties_df[col]
    display(evaluated.loc[valid_rows.index,
       ['canonical_smiles','MW','logP','QED','HD','HA','RB','TPSA','SAS']].head(12).round(3))
    print('QED among valid: mean =', round(float(evaluated.loc[valid_rows.index,'QED'].mean()),3))
else:
    print('No chemically valid molecules for property calculation.')

## 16. Reference thresholds and pass rates

Thresholds are taken from Ma'am's code: `logP < 5`, `MW < 500`, `HD < 5`, `HA < 10`, `RB < 10`, `TPSA < 140`, `SAS < 6`. These are **strict inequalities** as written in the original notebook, not silently relaxed. Besides regular pass percentages we calculate the reference notebook's uniqueness-weighted values, clearly labeled.

In [ ]:
thresholds = {'logP': 5, 'MW': 500, 'HD': 5, 'HA': 10,
              'RB': 10, 'TPSA': 140, 'SAS': 6}
for name, upper in thresholds.items():
    evaluated[f'{name}_pass'] = (evaluated['valid'].eq(1) & evaluated[name].lt(upper))
    if name == 'SAS' and not SAS_READY:
        evaluated[f'{name}_pass'] = pd.NA

print('Pass percentages among valid molecules:')
report = []
for name in thresholds:
    if name == 'SAS' and not SAS_READY:
        report.append((name, 'unavailable', 'unavailable'))
    else:
        hits = int(evaluated.loc[evaluated.valid == 1,f'{name}_pass'].sum())
        report.append((name, hits, f'{100*hits/len(valid_rows):.1f}%' if len(valid_rows) else 'N/A'))
display(pd.DataFrame(report, columns=['constraint','passed_count','among_valid_pct']))

## 17. Per-protein, per-run results (reference-style + Ames)

For proper CTDDG comparison, aggregate **per protein, then per independent run**; avoid treating multiple molecules from one protein as independent proteins. A 104-protein assertion is only appropriate if you actually generate for 104 held-out proteins using a *conditional* checkpoint.

In [ ]:
def summarize_group(g):
    N = int(g.requested_n.iloc[0]); good=g[g.valid==1]
    nu=good.canonical_smiles.nunique(); nv=len(good)
    scored_g=good[good.ames_prediction.notna()]
    n_scores=len(scored_g); pos=int(scored_g.ames_prediction.eq(1).sum())
    d={'requested':N,'exported':len(g),'valid_count':nv,
       'validity_pct':100*nv/N, 'unique_count':nu,
       'uniqueness_pct_ref':100*nu/N,
       'uniqueness_pct_valid':100*nu/nv if nv else np.nan,
       'ames_scored':n_scores,'ames_positive':pos,
       'ames_positive_pct_scored':100*pos/n_scores if n_scores else np.nan,
       'ames_negative_pct_scored':100*(n_scores-pos)/n_scores if n_scores else np.nan,
       'QED_mean_valid':good.QED.mean() if nv else np.nan,
       'QED_ref_sum_over_requested':good.QED.sum()/N}
    if 'pkcsm_ames' in g.columns:
        pk_sub = g.loc[g.pkcsm_ames.notna()]
        d['pkcsm_scored'] = len(pk_sub)
        d['pkcsm_ames_positive_pct_scored'] = (100*pk_sub.pkcsm_ames.eq(1).mean()
                                               if len(pk_sub) else np.nan)
    for key in thresholds:
        if key=='SAS' and not SAS_READY:
            d[f'{key}_pct_requested']=np.nan
            d[f'{key}_pct_valid']=np.nan
            d[f'{key}_maam_weighted_pct']=np.nan
            continue
        k=int(g.loc[g.valid==1,f'{key}_pass'].sum())
        d[f'{key}_pct_requested']=100*k/N
        d[f'{key}_pct_valid']=100*k/nv if nv else np.nan
        d[f'{key}_maam_weighted_pct']=100*(nu/N)*(k/N)
    return pd.Series(d)

grouped = (evaluated.groupby(group_cols).apply(summarize_group).reset_index())
display(grouped[['run','protein_id','validity_pct','uniqueness_pct_ref','ames_positive_pct_scored',
                 'logP_pct_requested','MW_pct_requested','QED_ref_sum_over_requested']].head(20).round(2))
print('Group count:',len(grouped))

In [ ]:
# Aggregate independent groups per run, then report descriptive means and standard deviations.
metric_columns = ['validity_pct','uniqueness_pct_ref','ames_positive_pct_scored',
                  'ames_negative_pct_scored','QED_ref_sum_over_requested'] + [
                  f'{k}_maam_weighted_pct' for k in thresholds]
per_run = grouped.groupby('run')[metric_columns].mean(numeric_only=True).reset_index()
print('Each run is averaged across its available proteins:')
display(per_run.head(12).round(3))
if len(per_run) > 1:
    stats = pd.DataFrame({'metric':metric_columns,
                          'mean_over_runs':[per_run[c].mean() for c in metric_columns],
                          'sd_over_runs':[per_run[c].std(ddof=0) for c in metric_columns]})
    display(stats.round(3))
else:
    print('One run only: no repeated-run SD. Set REPETITIONS=10 and generate 10 runs for paper-style stats.')

## 18. Check the **1,042 train / 104 test** requirement

These counts are counts of **distinct protein sequences**, not 1,042 training ligand rows / 104 generated molecules. The paper requires disjoint low-similarity protein sets. This notebook **cannot verify 20% sequence similarity or split integrity** from only checkpoint files; that requires the actual BindingDB split manifests and sequences.

For an unconditional pretraining result, the correct outcome here is **“not applicable”**. For conditional generation, check that 104 *distinct test protein IDs* appear and verify they match the held-out split.

In [ ]:
if MODEL_STAGE == 'pretrained_unconditional':
    print('Stage: unconditional pretraining — 1,042 / 104 protein-conditioned evaluation NOT APPLICABLE.')
    print('This notebook does not claim the resulting molecules target any protein.')
else:
    proteins = set(grouped.protein_id)
    proteins.discard('unconditional')
    print('Distinct generated target IDs:',len(proteins),'/',EXPECTED_TEST_PROTEINS)
    if len(proteins) != EXPECTED_TEST_PROTEINS:
        print('WARNING: incomplete/non-paper test protein coverage.')
    print('This tests ID COUNT only. Protein-disjoint train/test and sequence-similarity checks remain necessary.')

## 18A. Optional BindingDB split verification

If you supply the actual train and test CSV files used for protein-conditioned fine-tuning, check whether they contain 1,042 and 104 **unique amino-acid sequences** and share none. This is only a count/disjointness test, **not** the paper’s pairwise sequence-identity check. The second uploaded notebook does not specify these split CSV paths, so they are deliberately optional.

In [ ]:
def read_protein_sequences(path):
    data = pd.read_csv(path)
    choices = ['protein_sequence','protein_seq','sequence','target_sequence','fasta','protein']
    col = next((c for c in data.columns if str(c).lower().strip() in choices), None)
    if col is None:
        raise ValueError('No protein sequence column found. Available columns: '+str(data.columns.tolist()))
    vals = set(data[col].dropna().astype(str).str.replace(r'\s+','',regex=True).str.upper())
    vals.discard('')
    return vals

if BINDINGDB_TRAIN_CSV is None or BINDINGDB_TEST_CSV is None:
    print('Split verification unavailable: specify both BindingDB train and test CSV paths.')
else:
    tr = read_protein_sequences(BINDINGDB_TRAIN_CSV)
    te = read_protein_sequences(BINDINGDB_TEST_CSV)
    print('Distinct training sequences:',len(tr),'(paper target:',EXPECTED_TRAIN_PROTEINS,')')
    print('Distinct held-out sequences:',len(te),'(paper target:',EXPECTED_TEST_PROTEINS,')')
    print('Exact train/test sequence overlap:',len(tr & te))
    assert not tr & te, 'STOP: identical protein sequences occur in train and test.'
    if len(tr)!=EXPECTED_TRAIN_PROTEINS or len(te)!=EXPECTED_TEST_PROTEINS:
        print('WARNING: splits do not match paper counts; verify fold selection and preprocessing.')
    print('Not yet checked: pairwise sequence identity threshold and protein IDs used in generation.')

## 19. Graphs and failure diagnostics

Visualize three key diagnostics: generated validity; RF-predicted Ames class balance; and estimated Ames-positive probabilities. These are descriptive diagnostics, not experimental toxicity measurements.

In [ ]:
fig, ax=plt.subplots(figsize=(6,3.6))
ax.bar(['RDKit valid','Invalid exported','Not exported'],
       [int(evaluated.valid.sum()), int((1-evaluated.valid).sum()),
        int(validity_table.requested.sum())-len(evaluated)])
ax.set_ylabel('Number of requested molecules'); ax.set_title('MXNet output: validity accounting')
plt.tight_layout(); plt.show()

In [ ]:
if n_scored:
    fig,ax=plt.subplots(figsize=(5.5,3.4))
    ax.bar(['Ames-negative','Ames-positive'],[n_neg,n_pos])
    ax.set_ylabel('RF-scored valid molecules'); ax.set_title('Predicted Ames mutagenicity (RF)')
    plt.tight_layout(); plt.show()
    fig,ax=plt.subplots(figsize=(5.5,3.4))
    ax.hist(scored.ames_positive_probability, bins=15)
    ax.set_xlabel('Predicted P(Ames+)');ax.set_ylabel('Number of molecules')
    ax.set_xlim(0,1);ax.set_title('Ames probability distribution (RF)')
    plt.tight_layout();plt.show()
else:
    print('Skipping toxicity plots: no valid molecules were RF-scored.')

## 20. Save **all attempts**, metrics, and provenance without overwriting originals

The destination is a fresh timestamped directory. Files include detailed per-molecule invalid reasons, per-group and per-run scores, and a JSON audit summary with checkpoint hashes and denominator definitions. No checkpoint or input dataset is modified.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=False)
outputs = {'attempts_with_metrics.csv':evaluated,
           'valid_molecules_only.csv':evaluated.loc[evaluated.valid==1],
           'per_protein_run_metrics.csv':grouped,
           'per_run_metrics.csv':per_run,
           'source_count_audit.csv':counts}
for filename, frame in outputs.items():
    target=OUT_DIR/filename
    frame.to_csv(target,index=False)
    print('Saved',target.name,'rows=',len(frame))

In [ ]:
summary = {
 'stage': MODEL_STAGE,
 'source_mode': SOURCE_MODE,
 'generated_source': str(GENERATED_SOURCE),
 'checkpoint': str(MXNET_WEIGHTS),
 'checkpoint_sha256': sha256_file(MXNET_WEIGHTS),
 'atom_types_sha256': sha256_file(ATOM_TYPES_PATH),
 'config_sha256': sha256_file(CONFIG_PATH),
 'rf_model': str(RF_MODEL_PATH) if rf else None,
 'rf_positive_class': RF_POSITIVE_CLASS if rf else None,
 'pkcsm_input_csv': str(PKCSM_AMES_CSV) if PKCSM_AMES_CSV is not None else None,
 'pkcsm_scored_count': (int(evaluated.pkcsm_ames.notna().sum()) if 'pkcsm_ames' in evaluated else None),
 'pkcsm_ames_positive_pct_scored': (float(100*evaluated.loc[evaluated.pkcsm_ames.notna(), 'pkcsm_ames'].eq(1).mean())
    if 'pkcsm_ames' in evaluated and evaluated.pkcsm_ames.notna().any() else None),
 'rf_fingerprint': f'Morgan radius={MORGAN_RADIUS}, bits={MORGAN_BITS}',
 'requested_attempts': n_requested,
 'exported_rows':len(evaluated),
 'valid_count':int(evaluated.valid.sum()),
 'validity_pct_global': 100*int(evaluated.valid.sum())/n_requested,
 'unique_valid_count_global':int(valid_rows.canonical_smiles.nunique()),
 'rf_scored':n_scored,
 'ames_positive_count':n_pos if n_scored else None,
 'ames_positive_pct_scored':100*n_pos/n_scored if n_scored else None,
 'toxicity_method':'Random Forest predicted Ames mutagenicity; NOT pkCSM; NOT clinical toxicity',
 'validity_denominator':'per-run per-protein generator requested count; includes unexported attempts',
 'unexported_attempts':int(counts.not_exported.sum()),
 'n_distinct_exported_proteins': int(grouped.protein_id.nunique()),
 'reference_train_proteins':EXPECTED_TRAIN_PROTEINS,
 'reference_test_proteins':EXPECTED_TEST_PROTEINS,
 'comparison_to_conditional_104_protein_paper_applicable': MODEL_STAGE != 'pretrained_unconditional',
 'note':'Presence of 104 target IDs does not by itself verify leakage or protein split integrity.'
}
summary_path=OUT_DIR/'evaluation_summary.json'
summary_path.write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf8')
print('Saved',summary_path)
print('Chemical validity:',f"{summary['validity_pct_global']:.2f}%")
print('Predicted Ames-positive among scored:',
      f"{summary['ames_positive_pct_scored']:.2f}%" if n_scored else 'not evaluated')
print('Original files unchanged. Results:',OUT_DIR)

## Interpretation / limits of this evaluation

1. **Validity** is RDKit sanitization, computed with the same `valid / requested` principle as `Evaluation_metrics.ipynb`. It does not prove pharmacological feasibility or safety. A missing exported row counts against output yield but has unknown sanitization failure cause.
2. **Predicted toxicity** is the loaded RF's Ames mutagenicity prediction on *valid* generated molecules. The CTDDG paper's ~19% toxicity result uses **pkCSM** on a selected five-protein evaluation, so do **not** claim a numerical reproduction from this RF notebook. Labels must be checked against the RF's training data.
3. **Pretrained weights** may be *unconditional*. Only a verified **protein-conditioned** checkpoint and 104 held-out protein sequences can support the reported conditional evaluation. Check the architecture, checkpoint origin, and actual split file.
4. **Ma'am's unusual property formula** multiplies `unique / requested × property_pass_count / requested × 100`. Those figures are exposed as `*_maam_weighted_pct`; they are **not** the usual `pass / valid` percentages.
5. This notebook **does not dock molecules** (SMINA required) or run pkCSM. It also cannot generate raw molecular graphs without the matching MXNet model/sampler implementation, which was **not included** among the supplied files.
6. If you want invalid-valence **failure reasons**, export every graph attempt from the matching MXNet decoder, not just its successful canonical SMILES.